# Liora · ESCENA 001 «Mi primer maquillaje» · Kaggle (0 €)

La auténtica Liora Vale (**LIORA-VALE-0001**) con Wan 2.1 VACE 1.3B y pesos oficiales: 5 s, 480×832 y 16 fps. El primer fotograma y la referencia de identidad son el **maestro sellado**.

**Antes de ejecutar (una sola vez):**
1. *Add Input → Upload → New Dataset*, con visibilidad **Private**. Sube el archivo maestro exacto, el PNG de 941×1672 con SHA-256 `42a10b1f…`.
2. *Session options*: **GPU T4 x2** e **Internet On**.
3. **Save Version → Save & Run All**.

El cuaderno busca en `/kaggle/input` el archivo cuya huella coincide con el Identity Lock. Si no lo encuentra, se detiene. Nunca usa otra cara. La salida (`/kaggle/working/escena001`) no incluye la foto maestra.

In [ ]:
# 0) Detenerse antes de instalar nada si no hay GPU
import os, shutil, subprocess, pathlib
assert shutil.which('nvidia-smi'), 'No hay GPU NVIDIA asignada: Session options → Accelerator → GPU T4 x2.'
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout)
import torch
assert torch.cuda.is_available(), 'PyTorch no detecta CUDA.'
WORK = pathlib.Path('/kaggle/working/escena001'); WORK.mkdir(parents=True, exist_ok=True)
PRIV = pathlib.Path('/kaggle/temp_liora'); PRIV.mkdir(parents=True, exist_ok=True)   # privado: fuera de la salida publicada
os.environ.setdefault('HF_HOME', '/root/.cache/huggingface')

In [ ]:
# 1) Motor + Identity Lock desde main, y dependencias validadas
!rm -rf edge-funding && git clone -q --depth 1 --filter=blob:none --sparse https://github.com/alvarogavilan/edge-funding.git
!cd edge-funding && git sparse-checkout set liora/engine liora/identity && git log -1 --format='commit %H %cd'
os.chdir('edge-funding/liora/engine')
!pip install -q "diffusers==0.41.0" "transformers==5.20.1" "accelerate==1.15.0" "huggingface_hub==1.33.0" "tokenizers==0.23.3" "safetensors==0.8.0" ftfy sentencepiece imageio-ffmpeg
!pip install -q dlib-bin face_recognition_models==0.3.0 mediapipe && (apt-get -qq install -y libegl1 libgles2 > /dev/null 2>&1 || true)

In [ ]:
# 2) Localizar el MAESTRO por su huella (no por el nombre) y comprobarlo contra el Identity Lock
import hashlib, json
lock = json.load(open('../identity/identity-lock.json'))
want = lock['master']['sha256']
found = None
for f in pathlib.Path('/kaggle/input').rglob('*'):
    if f.is_file() and f.suffix.lower() in ('.png', '.jpg', '.jpeg', '.webp', '.heic') and hashlib.sha256(f.read_bytes()).hexdigest() == want:
        found = f; break
assert found, f'No está el maestro sellado ({want[:16]}…) en /kaggle/input. Sube el archivo exacto a un dataset PRIVADO. No se usa ninguna otra cara.'
MASTER = PRIV / 'maestro.png'; shutil.copyfile(found, MASTER)
print('Maestro verificado:', lock['identity_id'], want[:16], '…')
!python ../identity/identity_lock.py check-file "$MASTER"
!python -m liora_engine identity-enroll --pack "$PRIV/pack.json" --image "$MASTER" --official

In [ ]:
# 3) Pesos oficiales de Wan 2.1 VACE 1.3B, con la licencia leída de la ficha
import time
from huggingface_hub import snapshot_download, model_info
t0 = time.time()
MODEL = snapshot_download('Wan-AI/Wan2.1-VACE-1.3B-diffusers')
info = model_info('Wan-AI/Wan2.1-VACE-1.3B-diffusers')
print(MODEL, '| licencia:', getattr(info.card_data, 'license', None), '| commit', info.sha, '|', round(time.time() - t0), 's')
os.environ['LIORA_WAN21_VACE'] = MODEL

In [ ]:
# 4) ESCENA 001: hasta 3 intentos (semillas distintas). Cada intento pasa por la identidad de dlib, el control
#    fotograma a fotograma y la continuidad. Si falla, se rechaza y no se exporta.
!python -m liora_engine escena001 --official "$MASTER" --pack "$PRIV/pack.json" --work "$WORK" --seconds 5 --steps 30

In [ ]:
# 4b) Solo si salieron fotogramas inválidos (desbordamiento fp16): repetir en bfloat16
m = json.load(open(WORK / 'manifest.json'))
bad = any(not a.get('frame_health', {}).get('ok', True) for a in m['scenes'][0]['attempts'])
if m['status'] != 'completed' and bad:
    os.environ['LIORA_DTYPE'] = 'bfloat16'
    !python -m liora_engine escena001 --official "$MASTER" --pack "$PRIV/pack.json" --work "$WORK/bf16" --seconds 5 --steps 30

In [ ]:
# 5) Verificación independiente y archivos para Liora Studio (Montaje → Importar del motor)
import glob
from IPython.display import Video
mp = sorted(glob.glob(str(WORK / '**/manifest.json'), recursive=True))[-1]
m = json.load(open(mp)); sc = m['scenes'][0]
for a in sc['attempts']:
    print(a['attempt'], 'aceptado' if a.get('accepted') else 'RECHAZADO', '| identidad', a.get('identity', {}).get('verdict'), a.get('identity', {}).get('median'),
          '| fotogramas', a.get('identity', {}).get('frames'), '| continuidad', a.get('continuity', {}).get('verdict'), '| error', a.get('error'))
assert m['status'] == 'completed', 'Ningún intento cumple los criterios: no hay ESCENA 001 válida. No se presenta como Liora oficial.'
clip = pathlib.Path(sc['clip'])
if not shutil.which('ffprobe'):
    !apt-get -qq install -y ffmpeg > /dev/null 2>&1
probe = json.loads(subprocess.check_output(['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-count_frames', '-show_entries',
    'stream=codec_name,profile,pix_fmt,width,height,r_frame_rate,nb_read_frames,duration', '-of', 'json', str(clip)]))['streams'][0]
print(json.dumps({'archivo': clip.name, 'bytes': clip.stat().st_size, 'sha256': hashlib.sha256(clip.read_bytes()).hexdigest(), **probe,
                  'modelo': sc['attempts'][-1].get('params', {}).get('model'), 'candado': m['identity_lock']}, indent=2, ensure_ascii=False))
!ls -la "$(dirname {mp})/studio"
Video(str(clip), embed=True, width=300)

**Después:** descarga la carpeta `escena001/studio` desde *Output*. En Liora Studio, abre Montaje → Importar del motor. La escena pasa por el control facial del iPhone y por tu revisión visual (manos, ojos, piel y pincel); solo entonces aparece en Publicaciones como APTO PARA PUBLICAR.